In [2]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
COMMIT = '31e974b23b7e73b2f48b080b36865608efb879b8'
ROOT = Path('/content/e05_source')
SETUP_ENV = dict(os.environ, UV_PYTHON_INSTALL_DIR='/content/e05_python', UV_CACHE_DIR='/content/e05_uv_cache', UV_NATIVE_TLS='true', CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1', MPLCONFIGDIR='/content/e05_mpl', MPLBACKEND='Agg')
print(subprocess.check_output(['nvidia-smi'], text=True))
assert not ROOT.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ROOT)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',COMMIT],cwd=ROOT,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
source = {p.relative_to(ROOT).as_posix():p.read_text(encoding='utf-8') for folder in ('src','configs','experiments/configs') for p in (ROOT/folder).rglob('*') if p.suffix in {'.py','.json'}}
code_hash = hashlib.sha256(json.dumps(source,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
assert code_hash == 'cdea42adb930904fb0cefcd282d784bb3ac8caa311d3f1e12ef568622f9c12d6'
subprocess.run([sys.executable,'-m','pip','install','--target','/content/e05_tools','uv==0.12.7'],env=SETUP_ENV,check=True)
UV = Path('/content/e05_tools/bin/uv')
subprocess.run([str(UV),'python','install','3.11.14'],env=SETUP_ENV,check=True)
subprocess.run([str(UV),'venv','--seed','--python','3.11.14','/content/e05_venv'],env=SETUP_ENV,check=True)
PY = Path('/content/e05_venv/bin/python')
freeze = (ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock = Path('/content/e05_lock.txt')
lock.write_text('\n'.join(x for x in freeze if not x.startswith('-e '))+'\n')
r = subprocess.run([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'],env=SETUP_ENV,capture_output=True,text=True); print(r.stdout,r.stderr[-1500:]); r.check_returncode()
r = subprocess.run([str(UV),'pip','install','--python',str(PY),'-r',str(lock)],env=SETUP_ENV,capture_output=True,text=True); print(r.stdout,r.stderr[-1500:]); r.check_returncode()
subprocess.run([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'],env=SETUP_ENV,check=True)
subprocess.run([str(PY),'-m','pip','check'],env=SETUP_ENV,check=True)
print('E05 isolated environment provisioned:',COMMIT,code_hash,flush=True)

Mon Oct 12 11:07:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import uuid, json
assert Path('/content/drive/MyDrive').is_dir()
STORE = Path('/content/drive/MyDrive/Nano-Language-Model/E05_seed42') / ('pilot_'+uuid.uuid4().hex)
STORE.mkdir(parents=True,exist_ok=False)
OUT = Path('/content/e05_pilot')
print('Fresh E05 storage:',STORE)

Mounted at /content/drive
Fresh E05 storage: /content/drive/MyDrive/Nano-Language-Model/E05_seed42/pilot_e2a4d3081a4344559641bdacaac89e54


In [4]:
ORCH = Path('/content/e05_orchestration')
ORCH_COMMIT = '3aaaa911c69d06e2c9693f988edc8cb5b536abd6'
assert not ORCH.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ORCH)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',ORCH_COMMIT],cwd=ORCH,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ORCH,text=True).strip()
SETUP_ENV['PYTHONPATH'] = str(ORCH)
PREFLIGHT_CMD = [str(PY),str(ORCH/'scripts/pilot/e05_colab.py'),'preflight','--root',str(ROOT),'--output',str(OUT),'--store',str(STORE),'--quota-free-bytes','390460000000']
preflight_log = (STORE/'preflight.log').open('x')
preflight_proc = subprocess.Popen(PREFLIGHT_CMD,cwd=ROOT,env=SETUP_ENV,stdout=preflight_log,stderr=subprocess.STDOUT,start_new_session=True)
print('E05 CUDA/data/storage preflight PID:',preflight_proc.pid)

E05 CUDA/data/storage preflight PID: 2447


In [7]:
print('Preflight returncode:',preflight_proc.poll())
print((STORE/'preflight.log').read_text()[-2500:])
print('Qualification files:',[p.name for p in OUT.glob('*.json')])
print((OUT/'tests.json').read_text() if (OUT/'tests.json').exists() else 'Tests pending')
print((OUT/'activation_test.json').read_text() if (OUT/'activation_test.json').exists() else 'Activation test pending')
print((OUT/'preflight.json').read_text() if (OUT/'preflight.json').exists() else 'Storage qualification pending')

Preflight returncode: 0
E05 PREFLIGHT PASSED {"status": "passed", "experiment": "E05", "seed": 42, "authorized_research_updates": 5000, "source_commit": "31e974b23b7e73b2f48b080b36865608efb879b8", "code_hash": "cdea42adb930904fb0cefcd282d784bb3ac8caa311d3f1e12ef568622f9c12d6", "parameters": 1804800, "dependency_packages": 46, "actual_drive_quota_free_bytes": 390460000000, "quota_source": "Google Drive storage UI observed before launch", "smoke_checkpoint_bytes": 22767372, "conservative_artifact_budget_bytes": 3350479024, "smoke_verified_snapshot_count": 3, "fresh_restore_hash_equal": true, "fresh_cuda_smoke_inference": "To be        ", "restored_smoke_total_step": 4, "research_updates": 0, "storage": "/content/drive/MyDrive/Nano-Language-Model/E05_seed42/pilot_e2a4d3081a4344559641bdacaac89e54", "seconds": 67.805317532}

Qualification files: ['activation_test.json', 'environment.json', 'relu_audit.json', 'tests.json', 'hardware.json', 'preflight.json', 'dataset_verification.json']
{
  "

In [8]:
assert preflight_proc.poll() == 0
assert json.loads((OUT/'preflight.json').read_text())['status'] == 'passed'
assert not (OUT/'runs').exists() and not (STORE/'research_snapshots').exists()
RESEARCH_CMD = [str(PY),str(ORCH/'scripts/pilot/e05_colab.py'),'research','--root',str(ROOT),'--output',str(OUT),'--store',str(STORE)]
research_log = (STORE/'research.log').open('x')
research_proc = subprocess.Popen(RESEARCH_CMD,cwd=ROOT,env=SETUP_ENV,stdout=research_log,stderr=subprocess.STDOUT,start_new_session=True)
print('E05 seed42 research PID:',research_proc.pid,'authorized stop: 5000 completed updates')

E05 seed42 research PID: 3468 authorized stop: 5000 completed updates


In [20]:
runs = list((OUT/'runs').iterdir()) if (OUT/'runs').exists() else []
run_dir = runs[0] if len(runs) == 1 else None
events = [json.loads(x) for x in (run_dir/'events.jsonl').read_text().splitlines()] if run_dir and (run_dir/'events.jsonl').exists() else []
metrics = [e['metric'] for e in events if e.get('event') == 'step']
validations = [m for m in metrics if 'validation' in m]
snaps = [p for p in (STORE/'research_snapshots').glob('*') if (p/'export_manifest.json').exists()]
print(json.dumps({'returncode':research_proc.poll(),'run_id':run_dir.name if run_dir else None,'completed_step':metrics[-1]['step'] if metrics else 0,'full_validations':len(validations),'latest_validation':validations[-1] if validations else None,'durable_snapshots':len(snaps),'failures':[e for e in events if e.get('event')=='failure'],'log_tail':(STORE/'research.log').read_text()[-700:]},indent=2))

{
  "returncode": 0,
  "run_id": "E05_seed42_f36219171999455dbc35c098ad8458f9",
  "completed_step": 5000,
  "full_validations": 20,
  "latest_validation": {
    "step": 5000,
    "training_loss": 1.503288522362709,
    "training_tokens": 20480000,
    "training_bytes": 20480000,
    "unique_target_bytes": 1003848,
    "train_seconds": 1106.0735030009946,
    "tokens_per_second": 18515.948483020104,
    "steps_per_second": 4.52049523511233,
    "peak_gpu_bytes": 159081984,
    "gradient_norm": 0.5830808281898499,
    "amp_skipped_updates": 0,
    "lr": 0.0003,
    "accumulation": 8,
    "validation": {
      "loss": 1.5544990886635046,
      "perplexity": 4.73271526058592,
      "bpb": 2.2426681262812513,
      "nll_sum": 86296.4624080658,
      "scored_tokens": 55514,
      "scored_bytes": 55514,
      "conditioning_bytes": 256,
      "evaluated_text_bytes": 55770,
      "mode": "sliding",
      "context_length": 128
    },
    "generalization_gap": 0.05121056630079557
  },
  "durable_

In [21]:
assert research_proc.poll() == 0, 'Wait for completed 5000-update research run'
assert json.loads((OUT/'completion.json').read_text())['status'] == '5000_completed_optimizer_updates'
POST = Path('/content/e05_postprocess_source')
POST_COMMIT = '6004b26a131bcb7a3558cdb15507a22f6ec0dfcf'
assert not POST.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(POST)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',POST_COMMIT],cwd=POST,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=POST,text=True).strip()
FINISH_CMD = [str(PY),str(POST/'scripts/pilot/e05_colab.py'),'finish','--root',str(ROOT),'--output',str(OUT),'--store',str(STORE)]
finish_log = (STORE/'finish.log').open('x')
finish_proc = subprocess.Popen(FINISH_CMD,cwd=ROOT,env=SETUP_ENV,stdout=finish_log,stderr=subprocess.STDOUT,start_new_session=True)
print('Fresh CUDA recovery, full generation, archive publication PID:',finish_proc.pid,'additional optimizer updates: 0')

Fresh CUDA recovery, full generation, archive publication PID: 10613 additional optimizer updates: 0


In [24]:
print('Postprocessing returncode:',finish_proc.poll())
print('Recovery checkpoint files:',len(list((OUT/'independent_runs').rglob('step_*.pt'))))
print('CUDA state recovery:',json.loads((OUT/'cuda_recovery/recovery.json').read_text())['exact_state'] if (OUT/'cuda_recovery/recovery.json').exists() else 'Pending')
print('Analysis files:',[p.name for p in (OUT/'analysis').glob('*')])
print('Log tail:',(STORE/'finish.log').read_text()[-1200:])
print('Archive metadata:',(STORE/'archive_parts.json').read_text() if (STORE/'archive_parts.json').exists() else 'Pending')

Postprocessing returncode: 0
Recovery checkpoint files: 20
CUDA state recovery: {'model': True, 'optimizer': True, 'scheduler': True, 'scaler': True, 'rng': True, 'training_state': True}
Analysis files: ['validation.csv', 'summary.json', 'measured_history.json', 'figures', 'generation.json', 'generation.md']
Log tail: d0e29a9afcc0a9beeb10491daf2ee9c0feaa0272e07b1599a", "relu_audit.json": "106b775f8beda7c336fcced57ce30cc6eec07b8a20d938d23cdeccd7b5d2041f", "activation_diagnostics.json": "9b64c8405ba2a851828fba775b76a0537446149ee9fda4c7d6705748e1147b24", "tests.json": "c6cba918adf8d7dd1b3d6557e8c7dacd5291544eee0ca01a755b83aa4acc8172", "checkpoint_audit.json": "580e2b326d586cafde28f1416e0653c6fa5710b165de2f019947144129fc523e", "postprocessing_wall_time.json": "1ad165d3c501e371878c9eef62b225ba4077c357c1edb0fd83f50c301dd3c1c5", "hardware.json": "540e388a1bc05d639adcb1db7948e7265d795fdf40c565d84bf4826e33f85102", "snapshot_index.json": "0e71481ae6a6102babf2355fef26bae676a86b9f0c6bee0fded9cefc4

In [25]:
assert finish_proc.poll() == 0
DELIVERY_COMMIT = '39ba4ed1dc4950efd12f536e06012668ecdcf9e1'
subprocess.run(['git','fetch','--quiet','origin','research/e05-seed42-colab-pilot-20261012'],cwd=POST,check=True)
subprocess.run(['git','checkout','--quiet','--detach',DELIVERY_COMMIT],cwd=POST,check=True)
REPORT_SOURCE = POST/'report/pilots/E05_seed42'
REPORT_STORE = STORE/'report'
assert REPORT_SOURCE.is_dir() and not REPORT_STORE.exists()
import shutil, hashlib
shutil.copytree(REPORT_SOURCE,REPORT_STORE)
report_hashes = {p.relative_to(REPORT_SOURCE).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in REPORT_SOURCE.rglob('*') if p.is_file()}
assert all(hashlib.sha256((REPORT_STORE/n).read_bytes()).hexdigest() == h for n,h in report_hashes.items())
report_publication = {'delivery_commit':DELIVERY_COMMIT,'files':report_hashes,'readback_hashes_passed':True,'independent_retrieval_status':json.loads((REPORT_STORE/'evidence/independent_verification.json').read_text())['status']}
report_temp = STORE/('.report_manifest_'+uuid.uuid4().hex)
f = report_temp.open('x'); f.write(json.dumps(report_publication,indent=2)); f.flush(); os.fsync(f.fileno()); f.close()
os.rename(report_temp,STORE/'report_publication.json')
print('Final E05 report, comparison figures and independent receipts published; readback hashes passed:',len(report_hashes))

Final E05 report, comparison figures and independent receipts published; readback hashes passed: 46


In [26]:
s = json.loads((OUT/'analysis/summary.json').read_text())
v = json.loads((REPORT_STORE/'evidence/independent_verification.json').read_text())
print(json.dumps({'experiment':'E05 ReLU seed42','completed_updates':s['completed_optimizer_updates'],'full_validations':v['full_validations'],'best_and_final_step':s['best_validation']['step'],'validation_loss':round(s['final_validation']['loss'],8),'conditional_BPB':round(s['final_validation']['bpb'],8),'verified_checkpoint_states':v['checkpoint_count'],'genuine_generation_labels':v['generation_records'],'independent_integrity_and_inference':v['status'].endswith('_passed'),'report_files_hash_verified':len(report_publication['files']),'additional_optimizer_updates':v['additional_optimizer_updates']},indent=2))

{
  "experiment": "E05 ReLU seed42",
  "completed_updates": 5000,
  "full_validations": 20,
  "best_and_final_step": 5000,
  "validation_loss": 1.55449909,
  "conditional_BPB": 2.24266813,
  "verified_checkpoint_states": 20,
  "genuine_generation_labels": 100,
  "independent_integrity_and_inference": true,
  "report_files_hash_verified": 46,
  "additional_optimizer_updates": 0
}
